# Notebook 46 — Choosing an Open-Model Serving Engine

    ## Learning objectives

    - Compare Transformers, llama.cpp/MLX, Ollama, vLLM, SGLang, and managed HF endpoints by workload
- Build protocol conformance and quality/performance benchmark contracts
- Avoid architecture decisions based on feature checklists without hardware-specific evidence

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['httpx>=0.28']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 46.1 Start with the workload

Engine selection follows constraints: model architectures and formats, accelerator/CPU platform, single-user
versus concurrent traffic, prompt/output distributions, latency SLOs, adapters, structured output, tools,
multimodality, observability, isolation, and operator expertise. No engine is universally fastest. A feature may
exist but use a fallback path or be incompatible with a particular quantization.

Transformers offers direct research control. llama.cpp targets efficient local/cross-platform GGUF inference;
MLX is attractive on Apple silicon. Ollama wraps convenient local lifecycle and APIs. vLLM targets throughput-
oriented accelerator serving. SGLang combines a serving runtime with structured generation/programming features.
Managed Hugging Face endpoints outsource infrastructure lifecycle and expose several engines. TGI remains useful
in existing deployments but Hugging Face documents it as maintenance mode and recommends newer alternatives.


In [ ]:
engines = {
    "Transformers": {"local_debug":5, "gpu_throughput":2, "ops_simplicity":3},
    "llama.cpp/MLX": {"local_debug":4, "gpu_throughput":2, "ops_simplicity":4},
    "Ollama": {"local_debug":5, "gpu_throughput":2, "ops_simplicity":5},
    "vLLM": {"local_debug":2, "gpu_throughput":5, "ops_simplicity":2},
    "SGLang": {"local_debug":2, "gpu_throughput":5, "ops_simplicity":2},
    "managed endpoint": {"local_debug":1, "gpu_throughput":4, "ops_simplicity":5},
}
weights = {"local_debug":1, "gpu_throughput":3, "ops_simplicity":2}
print(sorted(((sum(v[k]*weights[k] for k in weights), name) for name,v in engines.items()), reverse=True))
print("Illustrative scores force priorities; replace with measured evidence.")


## 46.2 A compatibility matrix is versioned evidence

Test the exact model and engine image for tokenizer/chat-template behavior, context length, streaming, logprobs,
seed handling, stop strings, JSON Schema, tools, reasoning fields, embeddings, LoRA, quantizations, speculative
decoding, multimodal inputs, prefix caching, and cancellation. “OpenAI-compatible” means a partial protocol
surface, not identical validation, defaults, usage fields, errors, or output semantics.

Write conformance tests against a narrow internal client interface. Reject unknown response shapes, normalize
error classes deliberately, and keep engine-specific extensions behind feature flags. Pin model and tokenizer
commits rather than trusting mutable tags.


In [ ]:
required_contract = {"chat", "stream", "usage", "timeout", "cancel", "health"}
observed = {"Ollama": {"chat","stream","usage","timeout","health"},
            "vLLM": {"chat","stream","usage","timeout","cancel","health"}}
for engine, features in observed.items(): print(engine, "missing:", sorted(required_contract-features))


## 46.3 Performance methodology

Separate cold model load, prefill/TTFT, decode/inter-token latency, end-to-end percentiles, and aggregate prompt/
output tokens per second. Use realistic length distributions, arrival processes, concurrency, cancellations, and
output limits. Avoid coordinated omission: offered requests must remain represented when the server queues.
Warm up compilation and caches, then test long enough to expose memory pressure and thermal/autoscaling effects.

Hold model revision, precision/quantization, prompt rendering, decoding, hardware, and output quality constant.
Continuous batching may improve throughput while worsening an interactive request's tail latency. Report rejected
and timed-out requests; throughput from only successful short outputs is misleading.


In [ ]:
samples = [{"latency":1.0,"prompt":100,"output":50}, {"latency":1.8,"prompt":500,"output":100},
           {"latency":.9,"prompt":80,"output":40}, {"latency":4.2,"prompt":2000,"output":200}]
latencies = sorted(x["latency"] for x in samples)
percentile = lambda p: latencies[min(len(latencies)-1, int(p*len(latencies)))]
print({"p50": percentile(.5), "p95": percentile(.95),
       "output_tokens_per_wall_second": sum(x["output"] for x in samples)/max(x["latency"] for x in samples)})


## 46.4 Memory, batching, and topology

Capacity includes weights, KV cache for all live tokens, workspaces, graphs, adapter state, runtime overhead, and
fragmentation. Quantization reduces selected components, not all memory. Prefix caching benefits repeated exact
prefixes. Chunked prefill can reduce head-of-line blocking. Paged caches improve allocation but cannot create
physical memory.

Replicas increase independent throughput and fault isolation; tensor parallelism makes one model span devices but
adds communication. Pipeline and expert parallelism solve other placement problems. Optimize topology against
interconnect and workload. Autoscaling must account for multi-minute downloads and model load, cache warming,
draining, and minimum ready capacity.


In [ ]:
def kv_gib(layers, kv_heads, head_dim, live_tokens, bytes_per=2):
    return 2*layers*kv_heads*head_dim*live_tokens*bytes_per/2**30
for tokens in [8_000, 64_000, 256_000]: print(tokens, round(kv_gib(32, 8, 128, tokens), 2), "GiB KV")


## 46.5 Operations and security

Require readiness distinct from liveness, graceful drain, bounded queues, admission control by total tokens,
deadlines, cancellation propagation, per-tenant limits, and observable queue/prefill/decode stages. Pin container,
CUDA/driver, engine, model, tokenizer, templates, parsers, and launch arguments. Test OOM, worker loss, malformed
streams, slow clients, and rolling upgrades. A fallback must satisfy the same safety and data-location policy.

Put authenticated TLS ingress in front of model servers; isolate admin/metrics endpoints; constrain remote custom
code and dynamic adapters; validate schemas; cap payload/context/output; protect caches and logs; and audit model
downloads. The inference engine must not become the authorization layer for tools or retrieval.


## 46.6 Decision process

Shortlist engines that satisfy hard compatibility, platform, license, and security constraints. Run conformance
and frozen quality tests, then benchmark viable candidates on target hardware. Estimate operational cost and
failure recovery, perform a canary, and document the choice with expiry conditions. Keep the application portable
through contracts, not through avoiding engine-specific optimization entirely.

Reconsider when the model family, modality, quantization, traffic distribution, SLO, hardware, or team ownership
changes. The next lessons make two contrasting choices concrete: Ollama for approachable local serving and vLLM
for throughput-oriented accelerator deployments.


## 46.7 Select the engine with a compatibility matrix

Start with hardware and operating system, then model architecture, quantization format, multimodality, adapters, structured decoding, speculative decoding, distributed topology, API compatibility, observability, and operational maturity. Mark hard requirements separately from preferences. Benchmark only viable combinations with pinned versions. An engine that wins one throughput chart may not support the exact model, template, cache policy, or accelerator needed by the application.


In [ ]:
requirements={"cuda":True,"openai_compatible":True,"lora":True,"vision":False}; engines={"engine_a":{"cuda":True,"openai_compatible":True,"lora":True,"vision":True},"engine_b":{"cuda":True,"openai_compatible":True,"lora":False,"vision":True}}
print([name for name,caps in engines.items() if all(caps.get(k)==v for k,v in requirements.items())])


## 46.8 A fair serving bake-off

Use identical model and tokenizer revisions, templates, generation settings, precision, request traces, warmup, and hardware allocation. Replay realistic prompt/completion lengths and arrival patterns. Report quality parity, TTFT, inter-token latency, end-to-end percentiles, throughput, memory, error rate, cancellations, startup, and behavior near saturation. Verify API edge cases and output token accounting. Include installation, upgrades, debugging, metrics, and rollback in the decision rather than optimizing a single steady-state number.


In [ ]:
runs=[{"engine":"a","quality":.81,"p95":1.8,"rps":12,"errors":0},{"engine":"b","quality":.81,"p95":1.5,"rps":10,"errors":1}]; print(sorted(runs,key=lambda r:(r["errors"],r["p95"],-r["rps"])))


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [llama.cpp](https://github.com/ggml-org/llama.cpp)
- [SGLang documentation](https://docs.sglang.ai/)
- [vLLM documentation](https://docs.vllm.ai/)


## Exercises

    1. Write a protocol conformance suite and run it against two local endpoints.
2. Benchmark two engines with identical prompts, outputs, quantization, and quality gates.
3. Create an architecture decision record with explicit reconsideration triggers.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
